# Lesson 07 - Načrtovanje vzorca oblikovanja

Ta zvezek prikazuje **Načrtovanje vzorca oblikovanja** za AI agente z uporabo Microsoft Agent Framework.
Naučili se boste, kako razčleniti kompleksne zahteve za potovanje v strukturirane podnaloge, jih dodeliti specializiranim agentom 
in izvesti nastali načrt — vse s pomočjo strukturiranega izhoda, ki ga poganjajo Pydantic modeli.


## Namestitev


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Razčlenitev naloge

Razčlenitev naloge je jedro vzorca načrtovanja. Namesto da bi en sam agent obravnaval zahtevno zahtevo od začetka do konca, problem razdelimo na manjše, dobro definirane **podnaloge**. Vsaka podnaloga je dodeljena specializiranemu agentu (npr. leti, hoteli, aktivnosti) z jasnimi prednostmi in vrstnim redom odvisnosti.

Ta pristop prinaša več koristi:
- **Jasnost**: vsaka podnaloga ima eno samo odgovornost.
- **Vzporednost**: neodvisne podnaloge lahko potekajo hkrati.
- **Zanesljivost**: napake so izolirane na posamezne podnaloge.
- **Sledenje proračunu**: stroški so ocenjeni za vsako podnalogo in se seštejejo.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Ustvarjanje agenta za načrtovanje s strukturiranim izhodom

Agent za načrtovanje deluje kot **koordinator na recepciji**. Glede na visokoraven zahtevek za potovanje ustvari strukturiran `TravelPlan` — razčleni zahtevo na podnaloge, določi prioritete in prepozna odvisnosti, da lahko portir ali sloj za izvedbo opravi delo.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Izvajanje načrta s specializiranimi orodji

Ko je agent za recepcijo pripravil strukturiran načrt, ga izvede **agent za storitve**.
Vsako specializirano orodje obravnava eno kategorijo podnalog (leti, hoteli, aktivnosti). Agent za storitve
prehaja skozi podnaloge načrta po vrstnem redu odvisnosti in vsako posreduje ustreznemu orodju.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Povzetek

V tej lekciji ste spoznali **Načrtovalni vzorec** za AI agente:

1. **Razčlenitev naloge** — Agent načrtovanja na sprejemnem pultu razdeli zapleteno potovalno zahtevo na strukturirane podnaloge z uporabo Pydantic modelov in vsako dodeli specializiranemu agentu z določenimi prioritetami in odvisnostmi.
2. **Strukturiran izhod** — Z uporabo `response_format` agent vrne validiran objekt `TravelPlan` namesto prostega besedila, kar omogoča zanesljivo nadaljnjo obdelavo.
3. **Izvajanje načrta** — Agent koncierge prehaja skozi podnaloge z uporabo specializiranih orodij (`book_flight`, `reserve_hotel`, `book_activity`) za izvedbo načrta in poročanje rezultatov.

Ta vzorec loči *kaj storiti* (načrtovanje) od *kako to storiti* (izvajanje), zaradi česar so agenti bolj modularni, testabilni in lažje razširljivi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo storitve za avtomatski prevod [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, upoštevajte, da lahko avtomatizirani prevodi vsebujejo napake ali netočnosti. Izvirni dokument v izvorni jezik velja za avtoritativni vir. Za ključne informacije priporočamo strokovni človeški prevod. Nismo odgovorni za morebitna nesporazume ali napačne razlage, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
